# F1 Weekend Intelligence
ข้อมูลปี 2021–2023 · Notebook และเว็บใช้ `intelligence` ชุดเดียวกัน

เป้าหมาย: วิเคราะห์ lap จริงและประเมินการทำนายเวลาต่ำสุดของ Q1/Q2/Q3 ก่อน Qualifying โดยตรวจสอบข้อมูลทุกขั้นได้

เปิดผ่าน `docker compose --profile notebook up -d jupyter` แล้ว Run All ได้โดยไม่ดาวน์โหลดข้อมูลจาก FastF1

In [ ]:
from pathlib import Path
import os, sys
if not Path('intelligence').exists() and Path('f1_project/intelligence').exists():
    os.chdir('f1_project')
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from intelligence.data import Dataset, DATA, PRACTICE, clean_laps, hashes
from intelligence.ml import ARTIFACTS, NUMERIC, CATEGORIES, Preprocessor, ensure_artifacts, scenario
pd.set_option('display.max_columns', 40)

## 1. Raw extraction — ข้อมูลต้นทางอยู่ตรงไหน
CSV นี้เก็บคอลัมน์ที่เลือกจาก FastF1: results รายคน, practice ราย lap, weather ราย timestamp ข้อมูล FastF1 ผ่านการประมวลผลของ library แล้ว จึงไม่ใช่ raw feed ทุกคอลัมน์จากระบบจับเวลา

ขั้น export เปลี่ยน Timedelta เป็นวินาทีและเพิ่มปี/รายการแข่ง แต่ยังไม่ลบ lap, aggregate, impute, encode หรือ scale ไม่สร้างข้อมูลหรือเติมเวลาเอง

การเก็บข้อมูลใหม่เป็นคำสั่ง maintenance แยก (`data_collection.py` และ `intelligence.metadata`) ไม่อยู่ใน Run All

In [ ]:
raw = {name: pd.read_csv(DATA / 'raw' / filename, low_memory=False) for name, filename in {
    'qualifying_results': 'qualifying_results.csv',
    'practice_laps': 'practice_laps.csv',
    'qualifying_weather': 'qualifying_weather.csv'}.items()}
before_hashes = hashes(DATA / 'raw')
display(pd.DataFrame([{'table': k, 'rows': len(v), 'columns': len(v.columns), 'exact_duplicates': int(v.duplicated().sum())} for k,v in raw.items()]))
for name, table in raw.items():
    print(name)
    display(table.head(3))

## 2. Cleaning — ตรวจ missing, duplicate และ lap ที่ใช้ไม่ได้
รักษา CSV เดิม เพิ่ม flags พร้อมเลขแถวต้นทาง แล้วตัดตามลำดับ: duplicate → missing time/driver → nonpositive time → deleted → inaccurate → generated

แต่ละแถวมี removal reason เดียวเพื่อให้ยอดก่อน–หลังตรงกัน ส่วน flags อาจซ้อนกัน ห้ามนำยอด flags มาบวกเป็นจำนวนแถวที่ถูกตัด

In [ ]:
dataset = Dataset()
quality = dataset.quality()
display(pd.Series({k:v for k,v in quality.items() if isinstance(v, int)}).to_frame('rows'))
display(pd.Series(quality['removals'], name='sequential_removals').to_frame())
display(pd.Series(quality['overlapping_flags'], name='overlapping_flags').to_frame())
assert quality['raw_laps'] == quality['kept_laps'] + sum(quality['removals'].values())
display(dataset.laps.loc[~dataset.laps.usable, ['source_row','Driver','LapTime','reason','flag_inaccurate','flag_generated']].head(10))

## 3. Feature preparation — ใช้ข้อมูลที่รู้ก่อน Qualifying
เชื่อมข้อมูลด้วย event_id (ปี + round) แยกชื่อ Grand Prix ออกจาก circuit_id ของสนามจริง ใช้ SessionInfo StartDate/EndDate แบบ UTC ตรวจ session ที่จบก่อน Qualifying เท่านั้น ข้อมูลนี้คือ timestamp ที่ต้นทางรายงาน ไม่ใช่การวัดเวลาจบจริงใหม่

ตัวอย่าง British GP 2021: FP2 เกิดหลัง Qualifying จึงใช้แสดงย้อนหลังได้ แต่ไม่เป็น input ของโมเดล Weather ระหว่าง Qualifying ก็ไม่เป็น input

In [ ]:
display(dataset.sessions.loc[dataset.sessions.event_id.eq('2021-10'), ['session','start_utc','end_utc','before_qualifying']])
display(dataset.features.loc[dataset.features.event_id.eq('2021-10'), ['Driver',*PRACTICE,'QualiTime']].head())
assert dataset.features.loc[dataset.features.event_id.eq('2021-10'), 'FP2_Time'].isna().all()
print('Usable laps excluded by time cutoff:', quality['after_qualifying_laps'])
print('Pre-qualifying usable laps:', quality['pre_qualifying_laps'])
features = dataset.features.copy()
display(features[['event_id','Driver','Team','circuit_id',*PRACTICE,'QualiTime','predictable']].head())
display(features[PRACTICE].isna().sum().to_frame('missing_before_imputation'))

## 4. Split ก่อนเรียนรู้ preprocessing
2021: ฝึกเบื้องต้น · 2022 รอบ 1–11: เลือกโมเดล · 2022 รอบ 12–22: calibration ช่วง error · 2023: test สุดท้าย

แถวไม่มี target หรือไม่มี Practice ที่ใช้ได้ไม่ใช้ฝึก/ประเมิน เมื่อเลือกโมเดลแล้ว fit ใหม่บน 2021 + ชุด selection เท่านั้น

In [ ]:
eligible = features.loc[features.QualiTime.notna() & features.predictable].copy()
parts = {
 'training': eligible.loc[eligible.Year.eq(2021)],
 'selection': eligible.loc[eligible.Year.eq(2022) & eligible['round'].le(11)],
 'calibration': eligible.loc[eligible.Year.eq(2022) & eligible['round'].gt(11)],
 'test': eligible.loc[eligible.Year.eq(2023)]}
display(pd.DataFrame([{'partition': k, 'rows': len(v), 'events': v.event_id.nunique()} for k,v in parts.items()]))
for key, part in parts.items():
    for other, second in parts.items():
        if key != other: assert set(part.event_id).isdisjoint(second.event_id)

## 5. Imputation → Encoding → Scaling → Feature selection
Median imputation สำหรับ Practice; target encoding ของ Driver และ circuit_id เป็น out-of-fold โดย **แยกทั้ง event**; Team เป็น one-hot; numeric/encoded values ผ่าน StandardScaler; เลือกสูงสุด 15 features ด้วย f_regression

fit ทุกขั้นบน training เท่านั้น ข้อมูลที่ไม่รู้จักใช้ fallback จาก training ไม่ใช้ weather/Q1/Q2/Q3/Position เป็น feature

In [ ]:
processor = Preprocessor()
X_train = processor.fit_transform(parts['training'])
X_validation = processor.transform(parts['selection'])
print('Numeric features:', NUMERIC)
print('Target-encoded categories:', CATEGORIES)
print('Imputer training medians:', processor.imputer.statistics_)
print('Scaler training means:', processor.scaler.mean_)
print('Selected:', processor.selected)
display(pd.DataFrame(X_train, columns=processor.selected).head())
assert not pd.isna(X_train).any()
for fold in processor.fold_events:
    assert set(fold['fit']).isdisjoint(fold['validation'])
print('Every encoding fold separates complete events.')

## 6. Train และเลือกโมเดลด้วย validation
เปรียบเทียบ Practice baseline (เวลาที่ดีที่สุดก่อน Qualifying), Linear Regression, Random Forest เลือกด้วย validation RMSE และคงการเลือกนั้นไว้แม้ผล test จะแพ้ baseline

Artifacts จะถูกใช้ซ้ำถ้า checksum/code/dependency versions ไม่เปลี่ยน

In [ ]:
bundle = ensure_artifacts(dataset)
report = bundle['report']
print('Selected model:', report['selected_model'])
display(pd.DataFrame(report['selection']))
display(pd.DataFrame(report['test']))
print('Rows excluded from 2023 evaluation:', report['excluded_test_rows'])
assert bundle['selected'] == min(report['selection'], key=lambda r:r['rmse'])['model']

## 7. Calibration และ error รายสนาม
ช่วงอ้างอิง = prediction + percentile 5/95 ของ (actual − prediction) ใน calibration ปี 2022 แสดง coverage ที่วัดได้จริงในปี 2023 ไม่อ้างว่าเป็นช่วงรับประกัน 90%

In [ ]:
display(pd.Series(report['interval']).to_frame('value'))
display(pd.DataFrame(report['per_event']).head(9))
predictions = pd.read_csv(ARTIFACTS / 'predictions.csv')
active = predictions.loc[predictions.model.eq(bundle['selected'])]
fig, ax = plt.subplots(figsize=(9,4))
ax.scatter(active.QualiTime, active.prediction, s=16, alpha=.6)
limits = [min(active.QualiTime.min(), active.prediction.min()), max(active.QualiTime.max(), active.prediction.max())]
ax.plot(limits, limits, '--', color='grey')
ax.set(xlabel='Actual qualifying time (s)', ylabel='Prediction (s)', title='Untouched 2023 test')
plt.show()

## 8. ตรวจผล analytics กับเว็บ และทดลอง What-if
ตัวกรอง session/compound/driver ใช้ `Dataset` เดียวกันกับ API การเปรียบเทียบใช้ lap จริง ไม่ผสม best sectors ข้าม lap โดยไม่บอก

What-if เปิดสำหรับ 2023 และแก้เฉพาะ Practice ที่มีข้อมูลก่อน Qualifying ไม่มีข้อสรุปเชิงสาเหตุจากการเปลี่ยน input

In [ ]:
event_id = '2023-01'
comparison = dataset.compare(event_id, ['VER','HAM'], session='FP2', compound='SOFT')
display(pd.DataFrame([{k:v for k,v in row.items() if k != 'best_lap'} for row in comparison]))
display(dataset.filtered_laps(event_id, ['VER','HAM'], 'FP2', 'SOFT')[['lap_id','source_row','Driver','LapNumber','LapTime','Sector1Time','Sector2Time','Sector3Time']].head())
reference = scenario(dataset, bundle, event_id, 'VER', {})
changed = scenario(dataset, bundle, event_id, 'VER', {'FP2_Time': reference['inputs']['FP2_Time'] + .5})
display(pd.DataFrame([{'scenario':'original',**reference},{'scenario':'+0.5s FP2',**changed}])[['scenario','prediction','delta','lower','upper']])
assert hashes(DATA / 'raw') == before_hashes
print('Raw checksums unchanged after every step.')

## เปิดเว็บ
`http://localhost:8501` — Weekend, Compare, Prediction, Data & Method

ข้อจำกัด: ไม่ทราบเชื้อเพลิง/setup/run plan; ข้อมูล speed trap ไม่ใช่ telemetry ต่อเนื่อง; FastF1 และไฟล์ export อาจมีข้อมูลขาด ไม่มีการปลอมค่าเพื่อให้กราฟเต็ม

ตรวจระบบ: `docker compose run --rm test` · ทดสอบ browser: ดู README ส่วน verification